In [ ]:
import kagglehub
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

## Enable GPU Check

In [ ]:
print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

## Download FairFace Dataset
### Link: https://www.kaggle.com/datasets/aibloy/fairface/data

In [ ]:
print("Downloading dataset...")

path = kagglehub.dataset_download("aibloy/fairface")

print("Dataset downloaded at:", path)

## Load Dataset Labels

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

image_dir = os.path.join(path, "FairFace")
label_file = os.path.join(path, "FairFace/train_labels.csv")

df = pd.read_csv(label_file)

print("Dataset size:", len(df))
df.head()

## Convert Age Labels to Age Ranges

In [ ]:
def map_age(age):

    if age in ['0-2','3-9']:
        return "0-10"
    elif age == '10-19':
        return "11-20"
    elif age == '20-29':
        return "21-30"
    elif age == '30-39':
        return "31-40"
    elif age == '40-49':
        return "41-50"
    else:
        return "50+"

df["age_range"] = df["age"].apply(map_age)
df["filepath"] = df["file"].apply(lambda x: os.path.join(image_dir, x))
df.head()

## Data Generators

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=15,
    zoom_range=0.2,
    horizontal_flip=True
)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=df,
    x_col="filepath",
    y_col="age_range",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training"
)

val_generator = train_datagen.flow_from_dataframe(
    dataframe=df,
    x_col="filepath",
    y_col="age_range",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation"
)

## Build Transfer Learning Model

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

x = base_model.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)

output = tf.keras.layers.Dense(6, activation="softmax")(x)

model = tf.keras.Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## Model Training

In [ ]:
EPOCHS = 10

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS
)

## Save and Download Model

In [ ]:
from google.colab import files
model.save("age_model.keras")
print("Model saved!")
files.download("age_model.keras")